# Snapshot eval statistics tables — CSV + LaTeX export across eval settings

Companion to `sps_eval_catplot_cmp.ipynb`: it picks up the **same sources** (keep the `SOURCES`
block in sync) and exports **one statistics table per metric**, with one row per (algorithm,
variant) and **all eval settings side by side as columns** — the numbers behind that notebook's
stat glyphs. Which settings are tabled is set by `EVAL_CFGS` (default: all discovered).

`SEEDS` narrows every source to one seed (`42`), a list (`[42, 43]`, pooled into each cell) or
all (`None`). The seed matched against is the one each run recorded in its **own metadata**
(sidecar JSON `"seed"` for `eval-ray`/`eval-sb`, `args.json` `"resolved_seed"` for `eval-rbc`) —
never the directory name. Sources with no run on a wanted seed are skipped with a warning, so
narrowing to fan-out seeds drops sources that only ever ran the default seed. Keep `SEEDS` in
sync with the catplot notebook whenever both should report the same episodes.

The export cell writes, per metric, `<date>_<time>_out_<i>.{csv,tex}` next to this notebook —
the CSV and LaTeX body of one table share the same `i`:

- a **CSV** with a flat `<setting>_<stat>` header (full precision: count, mean, std, median,
  min, max per setting), and
- a **LaTeX table body** (`.tex`) — one `mean ± std` and one `median` column per setting,
  rounded to 2 decimals, `\midrule` between algorithm groups — meant to be `\input{}` into a
  hand-written booktabs `tabular` skeleton (a commented example sits at the top of the file), so
  rounding and ± assembly live in code while the LaTeX wrapper stays a thin shell.

The body **ends with `\bottomrule`**: do NOT add another one after the `\input{}`. LaTeX inserts
file-hook tokens at the end of an `\input{}`ed file which start a new table cell, so a following
`\midrule`/`\bottomrule` fails with *Misplaced `\noalign`*. Close the skeleton with
`\end{tabular}` directly.

> **Note:** `SOURCES` here still lists the 2026-07-14 battery-**capacity** snapshots, not this
> folder's `gen_lin_bat_pv_*` runs — sync it with the catplot notebook before exporting
> battery+PV tables.

**Main input:** `SOURCES`, `EVAL_CFGS`, `SEEDS`, `METRICS` and `TEX_VARIANT_LABELS` in the next
cell.

In [ ]:
# --- Main input ---------------------------------------------------------------------
# One source per (algorithm, training variant) — keep SOURCES in sync with
# snapshot_eval_catplot_algo_compare.ipynb (the catplot notebook draws ONE eval
# setting; this notebook tables the eval settings side by side).
# Keys per entry:
#   path          snapshot dir (absolute, or relative to the repo root)
#   eval-kind     "eval-ray" | "eval-sb" | "eval-rbc" — which runs/eval_* dirs to use
#   algo          first row key (also used in the export filenames)
#   variant       training-variant label, the second row key
#   runs          (optional) restrict to these runs/<name> directories; default: all
#   rbc-strategy  (eval-rbc only) which rule-based strategy to use

SOURCES = [
    # --- PPO -------------------------------------------------------------------
    {
        "path": "snapshots/20260719_171656_gen_lin_bat_pv_spec_ppo",
        "eval-kind": "eval-ray",
        "algo": "PPO",
        "variant": "spec",
    },
    {
        "path": "snapshots/20260719_171650_gen_lin_bat_pv_all_ppo",
        "eval-kind": "eval-ray",
        "algo": "PPO",
        "variant": "N-way gen.",
    },
    {
        "path": "snapshots/20260719_171702_gen_lin_bat_pv_sampled_ppo",
        "eval-kind": "eval-ray",
        "algo": "PPO",
        "variant": "gen. distribution",
    },
    # --- SAC -------------------------------------------------------------------
    {
        "path": "snapshots/20260719_171327_gen_lin_bat_pv_spec_sac",
        "eval-kind": "eval-ray",
        "algo": "SAC",
        "variant": "spec",
    },
    {
        "path": "snapshots/20260719_171314_gen_lin_bat_pv_all_sac",
        "eval-kind": "eval-ray",
        "algo": "SAC",
        "variant": "N-way gen.",
    },
    {
        "path": "snapshots/20260719_171333_gen_lin_bat_pv_sampled_sac",
        "eval-kind": "eval-ray",
        "algo": "SAC",
        "variant": "gen. distribution",
    },
    # snapshots/20260720_171650_gen_lin_bat_pv_all_dreamer
	# snapshots/20260720_171656_gen_lin_bat_pv_spec_dreamer
	# snapshots/20260720_171702_gen_lin_bat_pv_sampled_dreamer
    # --- DreamerV3 --
    {
		"path": "snapshots/20260720_171656_gen_lin_bat_pv_spec_dreamer",
		"eval-kind": "eval-ray",
		"algo": "DreamerV3",
		"variant": "spec",
	},
    {
		"path": "snapshots/20260720_171650_gen_lin_bat_pv_all_dreamer",
		"eval-kind": "eval-ray",
		"algo": "DreamerV3",
		"variant": "N-way gen.",
	},
    {
        "path": "snapshots/20260720_171702_gen_lin_bat_pv_sampled_dreamer",
        "eval-kind": "eval-ray",
        "algo": "DreamerV3",
        "variant": "gen. distribution",
    },
]


# Rule-based baselines: every strategy of this snapshot's eval_rbc run becomes one
# strip in the "RBC" x-axis group (drop entries here to trim the group). The
# rule-based eval is policy-independent — any snapshot with an eval_rbc run works.
RBC_SNAPSHOT = "snapshots/20260719_171314_gen_lin_bat_pv_all_sac"
RBC_STRATEGIES = (
    "do_nothing",
    "deficit_discharge",
    # "pv_surplus_charge",
    "self_coverage",
    "price_median",
    "price_median_scaled_0.25",
    "price_median_scaled_0.5",
    "price_median_scaled_0.75",
    "price_median_autarky",
)
SOURCES += [
    {"path": RBC_SNAPSHOT, "eval-kind": "eval-rbc", "algo": "RBC",
     "variant": strategy, "rbc-strategy": strategy}
    for strategy in RBC_STRATEGIES
]

# Which eval settings become the table columns. None = every setting discovered:
# held-out `eval_<i>` configs, numerically sorted (battery_lin_cap: eval_1 = BES
# 12 kWh, eval_2 = 16 kWh, eval_3 = 22 kWh); a flat STA layout appears as "default".
# Or an explicit tuple, e.g. ("eval_1", "eval_3").
EVAL_CFGS: tuple[str, ...] | None = None

# Eval seeds to keep, applied to every source after discovery:
#   SEEDS = None       keep every discovered seed (default)
#   SEEDS = 42         keep only the eval run seeded 42
#   SEEDS = [42, 43]   keep those two seeds, pooled into each table cell
# Matched against the seed every run recorded itself (sidecar JSON "seed" for
# eval-ray/eval-sb, args.json "resolved_seed" for eval-rbc), never the directory name.
# A source with no run on any wanted seed is skipped with a warning, so narrowing to the
# `_seed<N>` fan-out seeds drops sources that only ever ran the default seed. Keep in
# sync with the catplot notebook whenever both should report the same episodes.
SEEDS: int | list[int] | None = 42

# One exported table per metric: rows (algorithm, variant), columns the eval
# settings. Extend to also table e.g. "cum_E_kWh" and "cum_price_EUR".
METRICS = ("total_reward","cum_E_kWh","cum_price_EUR")

# Optional display names for the LaTeX export's variant column, keyed by the raw
# variant label. Only affects the .tex body — the CSV and the tables displayed in
# this notebook keep the raw keys. Values are inserted as RAW LaTeX (no escaping —
# math like $\times$ is allowed, escape %, &, _ yourself); unlisted variants keep
# their raw label, LaTeX-escaped. Shorter names keep the table inside the A4 text
# block without shrinking the font further.
TEX_VARIANT_LABELS: dict[str, str] = {
    "do_nothing": "do nothing",
    "deficit_discharge": "deficit discharge",
    "self_coverage": "self coverage",
    "price_median": "price median",
    "price_median_scaled_0.25": "price median $\\times$0.25",
    "price_median_scaled_0.5": "price median $\\times$0.5",
    "price_median_scaled_0.75": "price median $\\times$0.75",
    "price_median_autarky": "price median autarky",
}

In [12]:
import json
import re
from pathlib import Path

import pandas as pd

EVAL_KINDS = ("eval-ray", "eval-sb", "eval-rbc")

# A held-out eval-config result directory ends in `_eval_<i>` (e.g. battery_lin_cap_eval_1);
# training-config directories (e.g. battery_lin_cap_3) do not match and are skipped.
_EVAL_CFG_DIR_PATTERN = re.compile(r"(?:^|_)eval_(\d+)$")


# Walk upwards from the cwd until the directory containing snapshots/ is found, so the
# notebook runs both from thesis_eval/ and from the repo root.
def _find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "snapshots").is_dir():
            return candidate
    raise FileNotFoundError("No 'snapshots/' directory found upwards of the cwd — run inside the repo.")


REPO_ROOT = _find_repo_root()


# Maps a runs/eval_* directory to its eval kind — mirrors the run-id scheme
# tools/snapshot/submit_snapshot.py builds from --kind.
def _run_dir_eval_kind(run_dir: Path) -> str:
    if run_dir.name.startswith("eval_rbc_"):
        return "eval-rbc"
    if run_dir.name.startswith("eval_sb_"):
        return "eval-sb"
    return "eval-ray"


# `eval_<i>` for a held-out eval-config result dir name, None for anything else.
def _cfg_dir_setting(dir_name: str) -> str | None:
    match = _EVAL_CFG_DIR_PATTERN.search(dir_name)
    return f"eval_{match.group(1)}" if match else None


# Ray/SB3 eval CSVs carry a sidecar JSON (same stem) with the run's trial seed + algorithm.
def _sidecar_seed_and_algorithm(csv_path: Path) -> tuple[int | None, str | None]:
    sidecar = csv_path.with_suffix(".json")
    if not sidecar.is_file():
        return None, None
    try:
        payload = json.loads(sidecar.read_text())
    except (json.JSONDecodeError, OSError):
        return None, None
    return payload.get("seed"), payload.get("algorithm")


# Rule-based runs record resolved_seed in an args.json above episodes.csv (directly
# above the per-strategy dir in the flat layout, one level further up with per-config
# dirs) — walk up towards the run dir and take the first hit.
def _rbc_resolved_seed(csv_path: Path, run_dir: Path) -> int | None:
    for candidate_dir in csv_path.parents:
        args_path = candidate_dir / "args.json"
        if args_path.is_file():
            try:
                return json.loads(args_path.read_text()).get("resolved_seed")
            except (json.JSONDecodeError, OSError):
                return None
        if candidate_dir == run_dir:
            return None
    return None


# Discovers one record per eval CSV of `source` across ALL eval settings kept by
# EVAL_CFGS (None = no filter). Both layouts are globbed; every record is tagged with
# its setting ("default" for the flat STA layout, `eval_<i>` for held-out GEN config
# dirs).
def _discover_source_records(source: dict) -> list[dict]:
    eval_kind = source["eval-kind"]
    if eval_kind not in EVAL_KINDS:
        raise ValueError(f"'eval-kind' must be one of {EVAL_KINDS}, got {eval_kind!r}")

    snapshot_dir = Path(source["path"])
    snapshot_dir = snapshot_dir if snapshot_dir.is_absolute() else REPO_ROOT / snapshot_dir
    if not snapshot_dir.is_dir():
        raise FileNotFoundError(f"Snapshot directory not found: {snapshot_dir}")

    run_dirs = [
        run_dir for run_dir in sorted((snapshot_dir / "runs").glob("eval_*"))
        if _run_dir_eval_kind(run_dir) == eval_kind
    ]
    if wanted_runs := source.get("runs"):
        run_dirs = [run_dir for run_dir in run_dirs if run_dir.name in set(wanted_runs)]
    if not run_dirs:
        raise FileNotFoundError(f"No '{eval_kind}' run directories found under {snapshot_dir / 'runs'}")

    records = []
    for run_dir in run_dirs:
        if eval_kind == "eval-rbc":
            # flat: .../<strategy>/episodes.csv — nested: .../<strategy>/<cfg>_eval_<i>/episodes.csv
            candidates = [
                (csv_path, "default", csv_path.parent.name)
                for csv_path in sorted(run_dir.glob("eval_results/*_rule_based/*/episodes.csv"))
            ] + [
                (csv_path, _cfg_dir_setting(csv_path.parent.name), csv_path.parent.parent.name)
                for csv_path in sorted(run_dir.glob("eval_results/*_rule_based/*/*/episodes.csv"))
            ]
            strategies = sorted({strategy for _, _, strategy in candidates})
            strategy_wanted = source.get("rbc-strategy")
            if strategy_wanted is None and len(strategies) > 1:
                raise ValueError(
                    f"Source '{source['algo']} — {source['variant']}': multiple rule-based strategies "
                    f"{strategies} under {run_dir.name} — set 'rbc-strategy' to pick one."
                )
            for csv_path, setting, strategy in candidates:
                if setting is None or (strategy_wanted is not None and strategy != strategy_wanted):
                    continue
                records.append({
                    "algo": source["algo"], "variant": source["variant"], "snapshot_dir": snapshot_dir,
                    "run": run_dir.name, "setting": setting, "seed": _rbc_resolved_seed(csv_path, run_dir),
                    "eval_algorithm": strategy, "csv": csv_path,
                })
        else:
            candidates = [
                (csv_path, "default")
                for csv_path in sorted(run_dir.glob("eval_results/*_eval/eval_*.csv"))
            ] + [
                (csv_path, _cfg_dir_setting(csv_path.parent.name))
                for csv_path in sorted(run_dir.glob("eval_results/*_eval/*/eval_*.csv"))
            ]
            for csv_path, setting in candidates:
                if setting is None:
                    continue  # training-config result dir — not a held-out eval config
                seed, algorithm = _sidecar_seed_and_algorithm(csv_path)
                records.append({
                    "algo": source["algo"], "variant": source["variant"], "snapshot_dir": snapshot_dir,
                    "run": run_dir.name, "setting": setting, "seed": seed,
                    "eval_algorithm": algorithm, "csv": csv_path,
                })

    available = sorted({record["setting"] for record in records})
    if EVAL_CFGS is not None:
        records = [record for record in records if record["setting"] in set(EVAL_CFGS)]
    if not records:
        raise FileNotFoundError(
            f"Source '{source['algo']} — {source['variant']}': no eval CSVs on setting(s) "
            f"{EVAL_CFGS} under {snapshot_dir} (available: {available or 'none'})."
        )
    return records


_source_keys = [(source["algo"], source["variant"]) for source in SOURCES]
if len(_source_keys) != len(set(_source_keys)):
    raise ValueError(f"Every source needs a unique (algo, variant) pair; got {_source_keys}")

# SEEDS normalised to the set of wanted seeds (None = keep every discovered seed): a
# bare int selects one seed, any iterable of ints selects several.
SEED_FILTER: set[int] | None = (
    None if SEEDS is None else {int(seed) for seed in ([SEEDS] if isinstance(SEEDS, int) else SEEDS)}
)
if SEED_FILTER is not None and not SEED_FILTER:
    raise ValueError("SEEDS is empty — use None to keep every discovered seed, or name the seed(s) to keep.")

# Pool a source's eval runs — narrowed to SEED_FILTER when set — but keep only the
# first CSV per (setting, seed): the same checkpoint evaluated with the same seed on
# the same setting replays the identical eval episodes, so a duplicate would
# double-count them. Seed-less CSVs are kept and flagged while no seed filter is active
# (with one they cannot match, so they are dropped). A source left with nothing — eval
# run still pending, or no run on any wanted seed — is skipped with a warning.
eval_records: list[dict] = []
print(f"Repo root    : {REPO_ROOT}")
print(f"Eval settings: {'all discovered' if EVAL_CFGS is None else EVAL_CFGS}")
print(f"Eval seeds   : {sorted(SEED_FILTER) if SEED_FILTER is not None else 'all discovered (SEEDS = None)'}")
for source in SOURCES:
    try:
        discovered = sorted(_discover_source_records(source),
                            key=lambda record: (record["setting"], record["run"], record["csv"].name))
    except FileNotFoundError as error:
        print(f"\nWARNING: source '{source['algo']} — {source['variant']}' skipped — {error}")
        continue

    if SEED_FILTER is not None:
        for record in discovered:
            if record["seed"] is None:
                print(f"note: source '{source['algo']} — {source['variant']}': dropped {record['run']} / "
                      f"{record['csv'].name} on {record['setting']} (no seed resolvable — cannot match SEEDS)")
        seeds_available = sorted({record["seed"] for record in discovered if record["seed"] is not None})
        discovered = [record for record in discovered if record["seed"] in SEED_FILTER]
        if not discovered:
            print(f"\nWARNING: source '{source['algo']} — {source['variant']}' skipped — no eval CSVs with a "
                  f"seed in {sorted(SEED_FILTER)} (seeds available: {seeds_available or 'none'}) — "
                  "adjust SEEDS or the source.")
            continue

    kept, seen = [], set()
    for record in discovered:
        if record["seed"] is not None and (record["setting"], record["seed"]) in seen:
            print(f"note: source '{source['algo']} — {source['variant']}': dropped {record['run']} / "
                  f"{record['csv'].name} on {record['setting']} (duplicate seed {record['seed']})")
            continue
        if record["seed"] is None:
            print(f"WARNING: source '{source['algo']} — {source['variant']}': no seed resolvable for "
                  f"{record['csv'].name} — kept, duplicate eval runs undetectable")
        else:
            seen.add((record["setting"], record["seed"]))
        kept.append(record)
    eval_records.extend(kept)

    strategy = f", strategy={source.get('rbc-strategy')}" if source["eval-kind"] == "eval-rbc" else ""
    print(f"\nSource '{source['algo']} — {source['variant']}' ({source['eval-kind']}{strategy}): {source['path']}")
    for record in kept:
        print(f"  {record['setting']:>8}  seed {str(record['seed']):>5}  <-  {record['run']} / {record['csv'].name}")

if not eval_records:
    raise FileNotFoundError("No eval CSVs discovered for any source — nothing to table.")

# Row order = first-appearance order in SOURCES, restricted to sources with results.
_algos_with_records = {record["algo"] for record in eval_records}
_variants_with_records = {record["variant"] for record in eval_records}
algo_labels = list(dict.fromkeys(source["algo"] for source in SOURCES if source["algo"] in _algos_with_records))
variant_labels = list(dict.fromkeys(source["variant"] for source in SOURCES if source["variant"] in _variants_with_records))


# Column order: the flat "default" setting first (if present), then eval_<i> numerically.
def _setting_sort_key(setting: str) -> tuple[int, int]:
    match = re.fullmatch(r"eval_(\d+)", setting)
    return (0, 0) if match is None else (1, int(match.group(1)))


setting_labels = sorted({record["setting"] for record in eval_records}, key=_setting_sort_key)
print(f"\nTable columns (eval settings): {setting_labels}")


Repo root    : /hkfs/home/haicore/iai/dj0397/AdvBuildingGym
Eval settings: all discovered
Eval seeds   : [42]

Source 'PPO — spec' (eval-ray): snapshots/20260719_171656_gen_lin_bat_pv_spec_ppo
    eval_0  seed    42  <-  eval_20260721_103512 / eval_checkpoint_000058_20260721_110301.csv
    eval_1  seed    42  <-  eval_20260721_103512 / eval_checkpoint_000058_20260721_110326.csv
    eval_2  seed    42  <-  eval_20260721_103512 / eval_checkpoint_000058_20260721_110351.csv
    eval_3  seed    42  <-  eval_20260721_103512 / eval_checkpoint_000058_20260721_110415.csv

Source 'PPO — N-way gen.' (eval-ray): snapshots/20260719_171650_gen_lin_bat_pv_all_ppo
    eval_0  seed    42  <-  eval_20260721_103510 / eval_checkpoint_000058_20260721_105338.csv
    eval_1  seed    42  <-  eval_20260721_103510 / eval_checkpoint_000058_20260721_105406.csv
    eval_2  seed    42  <-  eval_20260721_103510 / eval_checkpoint_000058_20260721_105430.csv
    eval_3  seed    42  <-  eval_20260721_103510 / eval_check

## Output naming

Everything saved here lands next to the notebook as `<date>_<time>_out_<i>.<file format>`:
one timestamp per notebook run, `out_index` counting up once per exported table (its CSV and
`.tex` share that index), so no run overwrites an earlier one.

In [13]:
from datetime import datetime
from pathlib import Path

# VS Code exposes the notebook path; a plain Jupyter kernel starts in the notebook's directory.
OUT_DIR = Path(globals()["__vsc_ipynb_file__"]).parent if "__vsc_ipynb_file__" in globals() else Path.cwd()
OUT_RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
out_index = 0  # incremented once per saved artefact; reset by re-running this cell


def next_out_stem() -> Path:
    """Next '<date>_<time>_out_<i>' path stem in this notebook's directory.

    One index per exported table — its CSV and LaTeX body differ only in the suffix."""
    global out_index
    out_index += 1
    return OUT_DIR / f"{OUT_RUN_STAMP}_out_{out_index}"


## Load per-episode metrics

The kept CSVs are concatenated into one frame with a row per eval episode and the metrics as
columns, each row tagged with its `algo`, training `variant`, `eval_cfg` (the setting) and eval
`seed`. The preview below counts the episodes behind each table cell.

In [14]:
frames = []
for record in eval_records:
    csv_frame = pd.read_csv(record["csv"])
    present = [metric for metric in METRICS if metric in csv_frame.columns]
    missing = [metric for metric in METRICS if metric not in csv_frame.columns]
    if missing:
        print(f"note: source '{record['algo']} — {record['variant']}' {record['run']} lacks {missing} — "
              "skipped in those tables")
    frame = csv_frame[present].copy()
    frame["algo"] = record["algo"]
    frame["variant"] = record["variant"]
    frame["eval_cfg"] = record["setting"]
    frame["seed"] = record["seed"]
    frames.append(frame)

episodes = pd.concat(frames, ignore_index=True)
present_metrics = [metric for metric in METRICS if metric in episodes.columns]

# Episodes behind each table cell: rows (algo, variant), columns the eval settings.
episodes.groupby(["algo", "variant", "eval_cfg"]).size().unstack("eval_cfg").reindex(
    columns=setting_labels
).reindex(algo_labels, level="algo").reindex(variant_labels, level="variant")


eval_cfg                            eval_0  eval_1  eval_2  eval_3
algo      variant                                                 
PPO       spec                          10      10      10      10
          N-way gen.                    10      10      10      10
          gen. distribution             10      10      10      10
SAC       spec                          10      10      10      10
          N-way gen.                    10      10      10      10
          gen. distribution             10      10      10      10
DreamerV3 spec                          10      10      10      10
          N-way gen.                    10      10      10      10
          gen. distribution             10      10      10      10
RBC       do_nothing                    10      10      10      10
          deficit_discharge             10      10      10      10
          self_coverage                 10      10      10      10
          price_median                  10      10      10      10
          price_median_scaled_0.25      10      10      10      10
          price_median_autarky          10      10      10      10

## Per-strip statistics across the eval settings — display and export

One table per metric: rows = (algorithm, variant), columns = (eval setting, stat). The cell
below displays each table rounded to 3 decimals and writes the CSV (full precision) plus the
LaTeX body (2 decimals) next to this notebook. The variant column of the `.tex` uses the display
names from `TEX_VARIANT_LABELS`; the CSV and the displayed tables keep the raw variant keys.

In [15]:
from IPython.display import display

_STATS = ("count", "mean", "std", "median", "min", "max")


# Wide per-metric table: rows (algo, variant), columns (eval setting, stat).
def _metric_stats_wide(metric: str) -> pd.DataFrame:
    stats = episodes.groupby(["algo", "variant", "eval_cfg"])[metric].agg(list(_STATS))
    wide = stats.unstack("eval_cfg").swaplevel(axis=1)
    wide = wide.reindex(columns=pd.MultiIndex.from_product([setting_labels, _STATS]))
    return wide.reindex(algo_labels, level="algo").reindex(variant_labels, level="variant")


# LaTeX-escape the label columns (the RBC strategy names carry underscores).
def _latex_escape(text: str) -> str:
    return text.replace("&", r"\&").replace("%", r"\%").replace("_", r"\_")


# One table cell: "$mean \pm std$" at 2 decimals; bare mean when std is undefined
# (single-episode cell), "--" when a (row, setting) combination has no episodes.
def _latex_cell(mean: float, std: float | None = None) -> str:
    if pd.isna(mean):
        return "--"
    if std is not None and pd.notna(std):
        return f"${mean:.2f} \\pm {std:.2f}$"
    return f"${mean:.2f}$"


# Table body — rounding and ± assembly happen here, once, so the hand-written
# tabular skeleton that \input{}s this stays thin. \midrule separates the algorithm
# groups; the algorithm label is printed on its group's first row only. The file ends
# with \bottomrule ON PURPOSE: LaTeX inserts file-hook tokens at the end of an
# \input{}ed file, and after the last row's \\ those tokens start a new cell — a
# \bottomrule placed after \input{} in the skeleton therefore dies with
# "Misplaced \noalign". Inside the file it is safe; \end{tabular} may follow
# \input{} directly.
def _stats_latex_body(metric: str, wide: pd.DataFrame) -> str:
    lines = [
        f"% Auto-generated by snapshot_eval_stats_table_algo_compare.ipynb — booktabs body for '{metric}'",
        "% incl. trailing \\bottomrule (a \\bottomrule AFTER \\input{} would be 'Misplaced \\noalign').",
        "% Columns: algorithm & variant"
        + "".join(f" & {setting} mean+-std & {setting} median" for setting in setting_labels) + ".",
        "% Full precision, count, min and max: see the CSV with the same stem.",
        "% Verified skeleton:",
        f"%   \\begin{{tabular}}{{ll*{{{2 * len(setting_labels)}}}{{c}}}}",
        "%     \\toprule",
        "%     Algorithm & Variant & ... column heads ... \\\\",
        "%     \\midrule",
        "%     \\input{<this file>}",
        "%   \\end{tabular}",
    ]
    previous_algo = None
    for (algo, variant), row in wide.iterrows():
        if previous_algo is not None and algo != previous_algo:
            lines.append(r"\midrule")
        # TEX_VARIANT_LABELS values are raw LaTeX (no escaping); unmapped variants
        # fall back to their raw label, LaTeX-escaped.
        variant_cell = TEX_VARIANT_LABELS.get(variant, _latex_escape(str(variant)))
        cells = [_latex_escape(str(algo)) if algo != previous_algo else "", variant_cell]
        for setting in setting_labels:
            cells.append(_latex_cell(row[(setting, "mean")], row[(setting, "std")]))
            cells.append(_latex_cell(row[(setting, "median")]))
        lines.append(" & ".join(cells) + r" \\")
        previous_algo = algo
    lines.append(r"\bottomrule")
    return "\n".join(lines) + "\n"


for metric in present_metrics:
    wide = _metric_stats_wide(metric)
    export_frame = wide.copy()
    export_frame.columns = [f"{setting}_{stat}" for setting, stat in wide.columns]
    latex_body = _stats_latex_body(metric, wide)
    out_stem = next_out_stem()  # both formats of one table share the index
    export_frame.to_csv(f"{out_stem}.csv")
    Path(f"{out_stem}.tex").write_text(latex_body)
    print(f"saved {out_stem}.{{csv,tex}}  ({metric})")
    print(f"\n=== {metric} ===")
    display(wide.round(3))


saved /home/iai/dj0397/AdvBuildingGym/thesis_eval/6_gen_bat_cap_pv_power/20260901_110211_out_1.{csv,tex}  (total_reward)

=== total_reward ===


eval_0                                \
                                    count    mean    std  median    min   
algo      variant                                                         
PPO       spec                         10  11.704  4.360  12.381  4.913   
          N-way gen.                   10  11.666  3.786  13.212  4.291   
          gen. distribution            10  11.895  3.968  12.382  4.857   
SAC       spec                         10   9.900  2.456  10.957  4.538   
          N-way gen.                   10  11.883  3.335  11.958  5.058   
          gen. distribution            10  11.768  4.238  11.606  4.509   
DreamerV3 spec                         10   8.129  3.565   8.642 -0.620   
          N-way gen.                   10   8.275  3.080   8.954  1.942   
          gen. distribution            10   8.748  2.881   9.425  2.455   
RBC       do_nothing                   10   4.640  4.233   5.861 -2.838   
          deficit_discharge            10   3.092  3.112   3.455 -1.252   
          self_coverage                10   3.005  3.208   3.336 -1.394   
          price_median                 10   4.335  5.241   3.537 -3.386   
          price_median_scaled_0.25     10   7.375  4.209   7.437  1.014   
          price_median_autarky         10   5.594  4.401   6.438 -1.659   

                                           eval_1                         ...  \
                                       max  count    mean    std  median  ...   
algo      variant                                                         ...   
PPO       spec                      17.042     10  21.398  7.202  21.607  ...   
          N-way gen.                16.224     10  21.651  6.652  22.612  ...   
          gen. distribution         16.801     10  21.761  6.826  21.611  ...   
SAC       spec                      11.822     10  23.137  6.644  23.220  ...   
          N-way gen.                16.773     10  21.327  6.026  21.089  ...   
          gen. distribution         17.479     10  20.675  6.824  19.787  ...   
DreamerV3 spec                      11.574     10  18.078  5.160  18.067  ...   
          N-way gen.                11.741     10  15.923  5.805  16.283  ...   
          gen. distribution         12.077     10  17.531  5.285  17.305  ...   
RBC       do_nothing                11.084     10  11.649  7.312  12.294  ...   
          deficit_discharge          8.068     10   8.324  6.411   8.940  ...   
          self_coverage              8.019     10   8.255  6.483   8.895  ...   
          price_median              13.407     10  11.532  8.574  10.801  ...   
          price_median_scaled_0.25  14.742     10  15.979  7.157  15.578  ...   
          price_median_autarky      13.021     10  12.878  7.426  13.693  ...   

                                   eval_2                        eval_3  \
                                      std  median    min     max  count   
algo      variant                                                         
PPO       spec                      6.996  18.469  8.064  29.749     10   
          N-way gen.                6.774  19.660  7.309  29.204     10   
          gen. distribution         6.834  18.659  7.944  30.153     10   
SAC       spec                      7.060  20.003  8.006  30.551     10   
          N-way gen.                6.288  19.825  8.324  28.194     10   
          gen. distribution         6.848  17.754  7.826  28.369     10   
DreamerV3 spec                      5.933  15.812  5.925  25.115     10   
          N-way gen.                6.089  15.437  4.787  25.176     10   
          gen. distribution         5.980  14.595  4.830  25.301     10   
RBC       do_nothing                7.312  12.294  0.138  24.533     10   
          deficit_discharge         6.656   9.544  0.137  20.912     10   
          self_coverage             6.728   9.528 -0.019  20.874     10   
          price_median              8.310  11.379 -0.433  26.856     10   
          price_median_sc

saved /home/iai/dj0397/AdvBuildingGym/thesis_eval/6_gen_bat_cap_pv_power/20260901_110211_out_2.{csv,tex}  (cum_E_kWh)

=== cum_E_kWh ===


eval_0                                  \
                                    count    mean     std  median     min   
algo      variant                                                           
PPO       spec                         10  35.135  10.570  38.531  13.932   
          N-way gen.                   10  36.155  11.269  41.074  13.932   
          gen. distribution            10  36.155  11.269  41.074  13.932   
SAC       spec                         10  36.155  11.269  41.074  13.932   
          N-way gen.                   10  35.135  10.570  38.531  13.932   
          gen. distribution            10  35.135  10.570  38.531  13.932   
DreamerV3 spec                         10  34.950  11.918  40.391   8.803   
          N-way gen.                   10  36.118  11.349  41.074  13.569   
          gen. distribution            10  35.916  11.482  40.889  13.344   
RBC       do_nothing                   10  33.891  11.036  38.930  12.547   
          deficit_discharge            10  27.262  10.561  32.057   7.821   
          self_coverage                10  27.133  10.680  31.998   7.234   
          price_median                 10  30.425   8.940  31.084  12.547   
          price_median_scaled_0.25     10  32.579  10.238  35.498  12.547   
          price_median_autarky         10  33.891  11.036  38.930  12.547   

                                           eval_1                          \
                                       max  count    mean     std  median   
algo      variant                                                           
PPO       spec                      47.036     10  75.540  19.992  82.884   
          N-way gen.                47.478     10  76.900  20.826  86.623   
          gen. distribution         47.478     10  76.900  20.826  86.623   
SAC       spec                      47.478     10  76.900  20.826  86.623   
          N-way gen.                47.036     10  76.900  20.826  86.623   
          gen. distribution         47.036     10  75.540  19.992  82.884   
DreamerV3 spec                      46.931     10  75.730  20.069  84.868   
          N-way gen.                47.478     10  76.900  20.826  86.623   
          gen. distribution         47.478     10  76.177  20.634  83.827   
RBC       do_nothing                45.098     10  73.882  20.548  83.908   
          deficit_discharge         37.728     10  64.531  20.382  74.246   
          self_coverage             37.752     10  64.409  20.429  74.059   
          price_median              41.777     10  69.515  17.734  73.321   
          price_median_scaled_0.25  43.652     10  72.570  19.642  80.158   
          price_median_autarky      45.098     10  73.882  20.548  83.908   

                                    ...  eval_2                          \
                                    ...     std  median     min     max   
algo      variant                   ...                                   
PPO       spec                      ...  20.060  83.675  40.403  94.773   
          N-way gen.                ...  20.754  85.944  40.403  94.773   
          gen. distribution         ...  20.754  85.944  40.403  94.773   
SAC       spec                      ...  20.754  85.944  40.403  94.773   
          N-way gen.                ...  20.754  85.944  40.403  94.773   
          gen. distribution         ...  20.060  83.675  40.403  94.773   
DreamerV3 spec                      ...  20.393  85.430  39.951  93.672   
          N-way gen.                ...  20.756  85.828  40.403  94.773   
          gen. distribution         ...  20.218  83.756  39.716  94.773   
RBC       do_nothing                ...  20.548  83.908  37.773  92.836   
          deficit_discharge         ...  20.314  76.967  30.203  85.358   
          self_coverage             ...  20.361  76.781  30.203  85.351   
          price_median              ...  18.193  75.744  37.773  87.526   
          price_median_scaled_0.25  ...  19.642  80.158  37.7

saved /home/iai/dj0397/AdvBuildingGym/thesis_eval/6_gen_bat_cap_pv_power/20260901_110211_out_3.{csv,tex}  (cum_price_EUR)

=== cum_price_EUR ===


eval_0                                     \
                                    count   mean    std median    min    max   
algo      variant                                                              
PPO       spec                         10 -2.427  1.069 -2.438 -4.529 -0.760   
          N-way gen.                   10 -2.659  1.710 -2.332 -6.537 -0.663   
          gen. distribution            10 -2.538  1.156 -2.440 -4.465 -0.751   
SAC       spec                         10 -2.243  1.428 -1.860 -5.890 -0.701   
          N-way gen.                   10 -2.641  1.441 -2.179 -5.540 -0.782   
          gen. distribution            10 -2.511  1.204 -2.252 -4.646 -0.697   
DreamerV3 spec                         10 -1.817  1.099 -1.673 -3.855  0.096   
          N-way gen.                   10 -1.857  1.128 -1.743 -4.216 -0.300   
          gen. distribution            10 -1.980  1.226 -1.695 -4.685 -0.380   
RBC       do_nothing                   10 -1.152  1.198 -1.004 -3.567  0.439   
          deficit_discharge            10 -0.795  0.931 -0.622 -2.819  0.207   
          self_coverage                10 -0.781  0.947 -0.609 -2.816  0.230   
          price_median                 10 -1.110  1.393 -0.777 -3.857  0.559   
          price_median_scaled_0.25     10 -1.734  1.396 -1.279 -4.703 -0.157   
          price_median_autarky         10 -1.325  1.203 -1.177 -3.705  0.257   

                                   eval_1                       ... eval_2  \
                                    count   mean    std median  ...    std   
algo      variant                                               ...          
PPO       spec                         10 -4.508  1.941 -4.475  ...  1.973   
          N-way gen.                   10 -4.908  3.056 -4.229  ...  2.750   
          gen. distribution            10 -4.678  2.164 -4.443  ...  2.176   
SAC       spec                         10 -5.207  3.072 -4.623  ...  2.808   
          N-way gen.                   10 -4.865  3.120 -4.164  ...  2.835   
          gen. distribution            10 -4.440  2.091 -4.233  ...  2.062   
DreamerV3 spec                         10 -4.096  2.551 -3.628  ...  2.222   
          N-way gen.                   10 -3.564  2.133 -3.254  ...  2.181   
          gen. distribution            10 -3.940  2.363 -3.629  ...  2.043   
RBC       do_nothing                   10 -2.747  2.262 -2.248  ...  2.262   
          deficit_discharge            10 -2.038  1.987 -1.588  ...  2.058   
          self_coverage                10 -2.027  2.000 -1.577  ...  2.071   
          price_median                 10 -2.747  2.509 -1.987  ...  2.451   
          price_median_scaled_0.25     10 -3.659  2.532 -2.986  ...  2.466   
          price_median_autarky         10 -2.965  2.238 -2.473  ...  2.256   

                                                         eval_3                \
                                   median     min    max  count   mean    std   
algo      variant                                                               
PPO       spec                     -3.863  -7.328 -1.247     10 -2.879  1.149   
          N-way gen.               -3.652 -10.301 -1.130     10 -3.314  2.043   
          gen. distribution        -3.853  -7.807 -1.228     10 -2.983  1.210   
SAC       spec                     -3.961 -10.490 -1.238     10 -3.096  1.999   
          N-way gen.               -3.838 -10.835 -1.287     10 -2.791  1.257   
          gen. distribution        -3.843  -7.401 -1.210     10 -2.846  1.289   
DreamerV3 spec                     -3.241  -8.234 -0.916     10 -2.176  1.048   
          N-way gen.               -3.017  -7.970 -0.740     10 -2.061  1.103   
          gen. distribution        -3.092  -7.400 -0.747     10 -2.294  1.175   
RBC       do_nothing               -2.248  -7.386 -0.021     10 -1.152  1.198   
          deficit_discharge        -1.727  -6.543 -0.023     10 -0.737  0.861   
          self_coverage            -1.716  -6.539  0.00